In [ ]:
# import pkgs

import glob
import os
import pandas as pd
import numpy as np

import torch
from torch_geometric.data import Data
from torch_geometric.data import Dataset
from torch_geometric.loader import DataLoader

import warnings
warnings.filterwarnings('ignore')

# constant
SUN_NODE_ID = 0
PITCH_NODE_ID = 1
START_NODE_ID = 2
END_NODE_ID = 1850


# Test samples

In [ ]:
test_node_data = pd.read_csv('./data/시뮬레이션 CSV data_20250825/Output_1-42-86-0.65-3-1-9_case1_node.csv')
test_edge_data = pd.read_csv('./data/시뮬레이션 CSV data_20250825/Output_1-42-86-0.65-3-1-9_case1_edge.csv')
test_edge_data.rename(columns={'Distance(m)': 'Distance'}, inplace=True)

print(test_node_data.columns)
print(test_edge_data.columns)

In [ ]:
test_valid_edge_data = test_edge_data[test_edge_data['Edge_Type'] == "panel_to_panel"]
print(test_valid_edge_data.head())
print(test_valid_edge_data.shape)
print(len(set(test_valid_edge_data['Source'].tolist() + test_valid_edge_data['Target'].tolist())))



In [ ]:
y_1 = test_node_data['운동장_태양복사열'].median()
y_2 = test_node_data['관중석_1 태양복사열'].median()
y_3 = test_node_data['관중석_2 태양복사열'].median()
y_4 = test_node_data['관중석 합산 태양복사열'].median()

print(y_1, y_2, y_3, y_4)
print(test_node_data.columns)

In [ ]:
global_featues = {}
node_features = {}

node_features_type = {
    'Node_X': float,
    'Node_Y': float,
    'Node_Z': float,
    '열림각도': float,
    'Date_시간': str,
}
for row in test_node_data.itertuples():
    if row.Node_ID >= START_NODE_ID and row.Node_ID <= END_NODE_ID:
        node_features[row.Node_ID] = [row.Node_X, row.Node_Y, row.Node_Z, row.열림각도]
    if row.Node_ID == SUN_NODE_ID:
        global_featues[row.Node_ID] = [row.Node_X, row.Node_Y, row.Node_Z, row.Date_시간]

In [ ]:
edge_list = [test_valid_edge_data['Source'].tolist(), test_valid_edge_data['Target'].tolist()]
edge_list = torch.tensor(edge_list, dtype=torch.long)
edge_list


In [ ]:
edge_attr = [
    [row.Edge_Type, row.Distance]
    for row in test_valid_edge_data.itertuples()
]

In [ ]:
node_ids = sorted(node_features.keys()) 
id_map = {old: new for new, old in enumerate(node_ids)}
edge_index = torch.tensor([[id_map[n.item()] for n in edge_list[0]],
                           [id_map[n.item()] for n in edge_list[1]]])

x = torch.tensor([node_features[nid] for nid in node_ids], dtype=torch.float)
data = Data(
    x=x,
    edge_index=edge_list,
    edge_attr=edge_attr,
    y=y_1, # [y_1, y_2]
)

In [ ]:
DataLoader(
    [data, data, data], batch_size=1,
)